In [ ]:
#### This code is used to pull in temperature profile mean differences,  join with the predictor variables, export the tables for RF ####

In [ ]:
### Pull in Python Packages ###
import pandas as pd
import geopandas as gpd
import numpy as np
import glob
import os
import datetime as dt
from tqdm.notebook import tqdm

In [ ]:
##############################
##### Pull in the Dam Details  #####
##############################

In [ ]:
#### Get the Dams ####
Dams = pd.read_csv(r"F:\Insert_File_Path_Here\Clean_Dams_Site_Info.csv", engine = 'python') # Update this file path

## Get the List of the dams being used
Final_Dams_List = Dams["DamID"].unique().tolist()

#### Get more reservoir information #####
# Make an easier code for hydropower variables (joined from HILARRI) -- Updating the dictionary for simplicity
Dams["HydroCode"] = np.nan
damtype_dict = {'Hydropower dam associated with power plant; no reservoir': 'HDNR', 'Hydropower dam associated with reservoir and power plant': 'HDR', 'Hydropower dam only; no reservoir or power plant': 'HDNR','Hydropower dam associated with reservoir; no power plant':'HDR', None: 'NoHydro'} # dictionary
Dams['HydroCode'] = Dams['dataset'].map(damtype_dict)  # apply dictionary

# Pull in the list of non-hydro dams and classifications
NonHydroTypes = pd.read_csv(r"F:\Insert_File_Path_Here\Dam_Reservoir_Check.csv") # Update this file path

# Combine the codes
Selected_Dams = pd.merge(Dams, NonHydroTypes, left_on= "DamID", right_on = 'grod_id', how="left")
Selected_Dams["HydroCode"] = np.where((Selected_Dams['Manual_Check'] == 'NoReserv') , 'NoHydro-NR', Selected_Dams['HydroCode'])
Selected_Dams["HydroCode"] = np.where((Selected_Dams['Manual_Check'] == 'Reserv') , 'NoHydro-R', Selected_Dams['HydroCode'])

# Make a reservoir flag
Selected_Dams["Resv"] = np.nan
Selected_Dams["Resv"] = np.where((Selected_Dams['HydroCode'] == 'HDR'), 'Reservoir', Selected_Dams['Resv'])
Selected_Dams["Resv"] = np.where((Selected_Dams['HydroCode'] == 'NoHydro-R'), 'Reservoir', Selected_Dams['Resv'])
Selected_Dams["Resv"] = np.where((Selected_Dams['HydroCode'] == 'HDNR'), 'No Reservoir', Selected_Dams['Resv'])
Selected_Dams["Resv"] = np.where((Selected_Dams['HydroCode'] == 'NoHydro-NR'), 'No Reservoir', Selected_Dams['Resv'])

#### Clean up the columns ####
Selected_Dams = Selected_Dams[['DamID', 'Longitude', 'Latitude', 'Primary Purpose', 'NID Height (Ft)', 'Normal Storage (Acre-Ft)', 'Surface Area (Acres)',
       'Drainage Area (Sq Miles)', 'Max Discharge (Cubic Ft/Second)', 'Resv']]

In [ ]:
#######################################
####### Pull in the temperature data ########
#######################################

In [ ]:
# Downstream Diff -- Directly  Upstream
Avg_Diff_DIR = pd.read_csv(r"F:\Insert_File_Path_Here\Avg_Profile_Changes_Final.csv", engine = 'python') # Update this file path

In [ ]:
########################################
##### Pull in the Caravan/Basin Atlas data #####
########################################

In [ ]:
## First want to identify which of the basins we need to pull data from (Directly Upstream) ##
# Pull in the base SWORD nodes
SWORDNodes = gpd.read_file(r"F:\Insert_File_Path_Here\Obstructed_Nodes_Base_Clean.shp") # Update this file path
# Filter the nodes to just our dams of interest, and the 10km directly upstream
Up10_Node_Sub = SWORDNodes[(SWORDNodes["Assgn_dam"].isin(Final_Dams_List)) & (SWORDNodes["Dam_Dist"] >= -10000) & (SWORDNodes["Dam_Dist"] < 0)]

# Pull in the watersheds (Hydro7)
Hydro7 = gpd.read_file(r"F:\Insert_File_Path_Here\Basins_of_Interest_Hydro7.shp") # Update this file path

### Find the watershed upstream from each dam that has the majority of the 10km nodes in it##
# Join nodes to their intersecting Hydro7
NodeHy7Join = gpd.sjoin(Hydro7, Up10_Node_Sub, how='inner', predicate='contains')

# Group the points by watershed and dam
PointsPerHy7 = NodeHy7Join.groupby(['gauge_id','Assgn_dam']).agg({'Join_Node':['count']})
PointsPerHy7.columns = ["PointCount"]
PointsPerHy7 = PointsPerHy7.reset_index()

# Choose the Hydro7 (for each dam) with the majority points
MajorityPoints = PointsPerHy7.groupby("Assgn_dam")["PointCount"].idxmax()
UpstreamH7_byDam = PointsPerHy7.loc[MajorityPoints]
UpstreamH7_byDam_List = UpstreamH7_byDam["gauge_id"].unique().tolist()

# Preview
UpstreamH7_byDam

In [ ]:
## Export the data
UpstreamH7_byDam.to_csv(r'F:\Insert_File_Output_Path_Here\UpstreamHy7s_10km_byDam.csv') # Update this file path

In [ ]:
### Pull in the Caravan data for the given Hydro7 of interest ###
# Pull it attribute files 
caravan_attributes = pd.read_csv(r"F:\Insert_File_Path\attributes_caravan_Dams_HA_Hydro7.csv", engine = 'python', dtype={'gauge_id': object}) # Update this file path
caravan_hydroatlas = pd.read_csv(r"F:\Insert_File_Path\attributes_hydroatlas_Dams_HA_Hydro7.csv", engine = 'python', dtype={'gauge_id': object}) # Update this file path
caravan_misc = pd.read_csv(r"F:\Insert_File_Path\attributes_other_Dams_HA_Hydro7.csv", engine  = 'python', dtype={'gauge_id': object}) # Update this file path

## Pull in timeseries files 
Caravan_Timeseries_FileLoc = r"F:\Insert_File_Path\Caravan_P2_Outputs\timeseries\csv\Dams_HA_Hydro7" # Update this file path

# Get the list of all files
CSVFiles = glob.glob(os.path.join(Caravan_Timeseries_FileLoc, "*.csv")) 
# Filter the list of CSVs for only the relevant Hydro7s
CSVFiles_sub = [x for x in CSVFiles if any(y in x for y in UpstreamH7_byDam_List)]

# Loop through the files for each dam and make one dataframe
All_Caravan_Timeseries = pd.DataFrame()
for i in tqdm(range(len(CSVFiles_sub))):
    try:
        x = pd.read_csv(CSVFiles_sub[i], engine='python')
        Hy7ID = (CSVFiles_sub[i].split("\\")[8]).split('.')[0]
        x['gauge_id'] = Hy7ID
        All_Caravan_Timeseries = pd.concat([All_Caravan_Timeseries,x],axis=0)
    except pd.errors.EmptyDataError:
        print(CSVFiles_sub[i], " is empty and has been skipped.") # In case some of the CSV files are empty

# Preview the data
All_Caravan_Timeseries

In [ ]:
### "Static" and Monthly Avg Watershed Attributes by Site, Clean and ready for saving
Dam_WS_Att = pd.merge(UpstreamH7_byDam[['Assgn_dam', 'gauge_id']], caravan_attributes[['gauge_id', 'pet_mean', 'aridity', 'high_prec_freq', 'high_prec_dur']], on=['gauge_id'])

## Select relevant HydroBasin Atlas Attributes ##
HyBA_Attrib = caravan_hydroatlas.columns.tolist()

# Attributes of Interest 
Single_Attrib_OI = ['gauge_id','dis_m3_pyr', 'run_mm_syr','inu_pc_smx','dor_pc_pva','gwt_cm_sav', 'ele_mt_sav','for_pc_sse','urb_pc_sse']
Multiple_Attrib_Prefix = ['pre_mm_s', 'pet_mm_s', 'swc_pc_s']

Selected_Multi = Single_Attrib_OI[:]

for i in Multiple_Attrib_Prefix: 
    col_result = [x for x in HyBA_Attrib if x.startswith(i)]
    col_result.sort()
    Selected_Multi.extend(col_result)

# Filter the Caravan Data
Caravan_HyBA_Select = caravan_hydroatlas[Selected_Multi]

# Join with the other Caravan Data
Dam_WS_Att_All = pd.merge(Dam_WS_Att, Caravan_HyBA_Select, on = ['gauge_id'] )
Dam_WS_Att_All = Dam_WS_Att_All.reset_index(drop =True)
Dam_WS_Att_All = Dam_WS_Att_All.rename(columns={"Assgn_dam":'DamID', 'gauge_id': 'Hy7_ID'})
Dam_WS_Att_All

In [ ]:
print("Available 'Static' and Monthly Variables: ")
print(Dam_WS_Att_All.columns.tolist())

In [ ]:
## Export the more static watershed variables for each dam
Dam_WS_Att_All.to_csv(r'F:\Insert_File_Output_Path\UpWSAtt_byDam.csv') # Update this file path

In [ ]:
### Daily Avg Watershed Attributes by Site 
Daily_Dam_WS_Att_All = pd.merge(UpstreamH7_byDam[['gauge_id', 'Assgn_dam']], All_Caravan_Timeseries, on=['gauge_id'], how='outer')
Daily_Dam_WS_Att_All = Daily_Dam_WS_Att_All[['gauge_id', 'Assgn_dam', 'date', 'volumetric_soil_water_layer_1_mean']]

print("Available Daily Variables: ")
print(Daily_Dam_WS_Att_All.columns.tolist())

In [ ]:
### Filter the data to only the days that each dam has
# Get Daily Profiles
Daily_DS_Avg = pd.read_csv(r"F:\Insert_File_Path\Avg_Profile_Changes_Final.csv", engine = 'python', parse_dates = ['Date'])  # Update this file path

# Clean up some columns
DailyProfiles_Dam = Daily_DS_Avg[['Date', 'Assgn_dam']]
Daily_Attrib = Daily_Dam_WS_Att_All[:]
Daily_Attrib['Date'] = Daily_Attrib['date'].astype('<M8[ns]')

# Join the datasets
Daily_Dam_WS_Att_Prof = pd.merge(DailyProfiles_Dam, Daily_Attrib, on= ['Date','Assgn_dam'], how = "inner")
Daily_Dam_WS_Att_Prof

In [ ]:
### Export the CSV Files Needed ### 
Daily_Dam_WS_Att_Prof.to_csv(r'F:\Insert_File_Output_Path_Here\Daily_UpWSAtt_byDam_Prof.csv') # Update this file path

In [ ]:
###################################
##### Get the coincident RTMA info #####
###################################

In [ ]:
### Pull in RTMA Data -- process was run in seperate notebooks and output a folder with several CSV files
## RTMA file location
RTMA_FileLoc = r"F:\Insert_File_Path_Here\RTMA" # Update this file path

# Get the list of all files
RTMA_CSVFiles = glob.glob(os.path.join(RTMA_FileLoc, "*.csv")) 

# Loop through the all the RTMA files and make one dataframe
RTMA_Data = pd.DataFrame()
for i in tqdm(range(len(RTMA_CSVFiles))):
    try:
        x = pd.read_csv(RTMA_CSVFiles[i], engine='python')
        RTMA_Data = pd.concat([RTMA_Data,x],axis=0)
    except pd.errors.EmptyDataError:
        print(RTMA_CSVFiles[i], " is empty and has been skipped.") # In case some of the CSV files are empty

## Preview
RTMA_Data
 # Note: this is a csv of all possible nodes for all dams, on any of the days that the given days has surface temperature data

In [ ]:
### Select the RTMA values for our dams of interest
RTMA_Dam_Select = RTMA_Data[RTMA_Data['Assgn_dam'].isin(Final_Dams_List)]
RTMA_Dam_Select['Date'] = pd.to_datetime(RTMA_Dam_Select['date'], format ='%Y%m%d')

# Clean up the rows
RTMA_Dam_Select = RTMA_Dam_Select[[ 'Assgn_dam', 'Dam_Dist', 'Dam_Flag', 'Join_Node', 'lakeflag', 'node_id', 'WIND', 'Date']]
RTMA_Dam_Select['Assgn_dam'] = RTMA_Dam_Select['Assgn_dam'].astype(float)

# Get the days of interest for the profiles used
RTMA_Data_Filtered = pd.merge(RTMA_Dam_Select, DailyProfiles_Dam, on= ['Date','Assgn_dam'], how = "inner")

# Get Up/Ds 
RTMA_Data_Filtered["Up_Ds"] = np.nan
RTMA_Data_Filtered["Up_Ds"] = np.where((RTMA_Data_Filtered['Dam_Dist'] > 0) , 'Downstream', RTMA_Data_Filtered["Up_Ds"])
RTMA_Data_Filtered["Up_Ds"] = np.where((RTMA_Data_Filtered['Dam_Dist'] < 0) , 'Upstream', RTMA_Data_Filtered["Up_Ds"])

# Get Up/Ds waterbody type
RTMA_Data_Filtered['Up_Ds_Grp']  = np.nan
RTMA_Data_Filtered['Up_Ds_Grp'] = np.where((RTMA_Data_Filtered['Up_Ds'] == 'Downstream') & (RTMA_Data_Filtered['lakeflag'] == 0) , 'Downstream River', RTMA_Data_Filtered['Up_Ds_Grp'])
RTMA_Data_Filtered['Up_Ds_Grp'] = np.where((RTMA_Data_Filtered['Up_Ds'] == 'Downstream') & (RTMA_Data_Filtered['lakeflag'] > 0) , 'Downstream Reservoir', RTMA_Data_Filtered['Up_Ds_Grp'])
RTMA_Data_Filtered['Up_Ds_Grp'] = np.where((RTMA_Data_Filtered['Up_Ds'] == 'Upstream') & (RTMA_Data_Filtered['lakeflag'] == 0) , 'Upstream River', RTMA_Data_Filtered['Up_Ds_Grp'])
RTMA_Data_Filtered['Up_Ds_Grp'] = np.where((RTMA_Data_Filtered['Up_Ds'] == 'Upstream') & (RTMA_Data_Filtered['lakeflag'] > 0) , 'Upstream Reservoir', RTMA_Data_Filtered['Up_Ds_Grp'])

# Get the Dam Dist in km
RTMA_Data_Filtered['Dam_Dist_km'] = RTMA_Data_Filtered['Dam_Dist'] /1000

# Get the upstream average (10km)
RTMA_10km = RTMA_Data_Filtered[(RTMA_Data_Filtered['Dam_Dist_km'] >= -10) & (RTMA_Data_Filtered['Dam_Dist_km'] <= 0)]
RTMA_10km_avg = RTMA_10km.groupby(['Assgn_dam', 'Date']).agg({"WIND": 'mean'})
RTMA_10km_avg.columns = ["AvgUp_Wind"]
RTMA_10km_avg = RTMA_10km_avg.reset_index()

# Preview 
RTMA_10km_avg

In [ ]:
## Export the Data
RTMA_10km_avg.to_csv(r'F:\Insert_File_Output_Path_Here\RTMA_10km_WT_Avg.csv') # Update this file path

In [ ]:
##########################################
##### Get the Avg Up/DS Width information  #####
##########################################

In [ ]:
## Get the Nodes
Profile_Nodes_Width = pd.read_csv(r"F:\Insert_File_Path_Here\Combined_Temps_Nodes.csv", 
                                engine='python', usecols= ['Join_Node', 'Month', 'Day', 'Year', 'Avg_RWC_Wid', 'lakeflag', 'Assgn_dam', 'Dam_Flag', 'Up_Ds', 'Dam_Dist', 'Dam_Dist_km'])

 # Filter the nodes to the dams with information, and points wider than 100m
Profile_Nodes_Filtered = Profile_Nodes_Width[Profile_Nodes_Width["Assgn_dam"].isin(Final_Dams_List)]
Profile_Nodes = Profile_Nodes_Filtered[Profile_Nodes_Filtered['Avg_RWC_Wid']>= 100]

# Flag the waterbody type up/ds
Profile_Nodes['Up_Ds_Grp']  = np.nan
Profile_Nodes['Up_Ds_Grp'] = np.where((Profile_Nodes['Up_Ds'] == 'Downstream') & (Profile_Nodes['lakeflag'] == 0) , 'Downstream River', Profile_Nodes['Up_Ds_Grp'])
Profile_Nodes['Up_Ds_Grp'] = np.where((Profile_Nodes['Up_Ds'] == 'Downstream') & (Profile_Nodes['lakeflag'] > 0) , 'Downstream Reservoir', Profile_Nodes['Up_Ds_Grp'])
Profile_Nodes['Up_Ds_Grp'] = np.where((Profile_Nodes['Up_Ds'] == 'Upstream') & (Profile_Nodes['lakeflag'] == 0) , 'Upstream River', Profile_Nodes['Up_Ds_Grp'])
Profile_Nodes['Up_Ds_Grp'] = np.where((Profile_Nodes['Up_Ds'] == 'Upstream') & (Profile_Nodes['lakeflag'] > 0) , 'Upstream Reservoir', Profile_Nodes['Up_Ds_Grp'])

# Fix Padded Zeros Issue
Profile_Nodes['Month_Pad'] = Profile_Nodes['Month'].astype(str)
Profile_Nodes['Month_Pad'] = Profile_Nodes['Month_Pad'].str.pad(width=2, side='left', fillchar='0')
Profile_Nodes['Day_Pad'] = Profile_Nodes['Day'].astype(str)
Profile_Nodes['Day_Pad'] = Profile_Nodes['Day_Pad'].str.pad(width=2, side='left', fillchar='0')

# Create columns for index
Profile_Nodes['Date'] = Profile_Nodes[['Year', 'Month_Pad', 'Day_Pad']].apply(lambda row: '-'.join(row.values.astype(str)), axis=1)
Profile_Nodes['Date'] = pd.to_datetime(Profile_Nodes['Date'], format ='%Y-%m-%d')

## Filter to the profiles needed
Profile_Nodes_Daily =  pd.merge(Profile_Nodes, DailyProfiles_Dam, on= ['Date','Assgn_dam'], how = "inner")

In [ ]:
## Get the Avg Width Values and Diff for the profiles  ##
# Of The Profiles -- Avg Diff Up Vs Ds 
Upstream_Dir10_Wid = Profile_Nodes_Daily[(Profile_Nodes_Daily['Up_Ds'] == "Upstream") & (Profile_Nodes_Daily["Dam_Dist_km"] >= -10)]
Upstream_Avgs_Dir10_Wid = Upstream_Dir10_Wid.groupby(['Month','Day','Year','Assgn_dam']).agg({'Avg_RWC_Wid': ['mean']})
Upstream_Avgs_Dir10_Wid.columns = ['Up_Avg_Wid']
Upstream_Avgs_Dir10_Wid = Upstream_Avgs_Dir10_Wid.reset_index()

Downstream_Riv_D10_Wid = Profile_Nodes_Daily[(Profile_Nodes_Daily['Up_Ds_Grp'] == 'Downstream River') & (Profile_Nodes_Daily["Dam_Dist_km"] <= 10)]
Downstream_Avgs_Riv_D10_Wid = Downstream_Riv_D10_Wid.groupby(['Month','Day','Year','Assgn_dam']).agg({'Avg_RWC_Wid': ['mean']})
Downstream_Avgs_Riv_D10_Wid.columns = ['Ds_Avg_Wid']
Downstream_Avgs_Riv_D10_Wid = Downstream_Avgs_Riv_D10_Wid.reset_index()

Profile_Avgs_Dir10_Wid = pd.merge(Upstream_Avgs_Dir10_Wid, Downstream_Avgs_Riv_D10_Wid, on=['Month','Day', 'Year','Assgn_dam'], how='outer' )

Profile_Avgs_Dir10_Wid['Width_Diff'] = Profile_Avgs_Dir10_Wid['Ds_Avg_Wid'] - Profile_Avgs_Dir10_Wid['Up_Avg_Wid'] 

# Preview
Profile_Avgs_Dir10_Wid

In [ ]:
## Export the data
Profile_Avgs_Dir10_Wid.to_csv(r'F:\Insert_File_Output_Path_Here\Dir_Widths_byDam.csv') # Update this file path

In [ ]:
### From here, we need to get the data into a table that has Date(index) and then column for each dam's ds delta
# Get a proper date column
Dir10_Wid_Diff = Profile_Avgs_Dir10_Wid[:]

# Fix Padded Zeros Issue
Dir10_Wid_Diff['Month_Pad'] = Dir10_Wid_Diff['Month'].astype(str)
Dir10_Wid_Diff['Month_Pad'] = Dir10_Wid_Diff['Month_Pad'].str.pad(width=2, side='left', fillchar='0')
Dir10_Wid_Diff['Day_Pad'] = Dir10_Wid_Diff['Day'].astype(str)
Dir10_Wid_Diff['Day_Pad'] = Dir10_Wid_Diff['Day_Pad'].str.pad(width=2, side='left', fillchar='0')

# Create columns for index
Dir10_Wid_Diff['Date'] = Dir10_Wid_Diff[['Year', 'Month_Pad', 'Day_Pad']].apply(lambda row: '-'.join(row.values.astype(str)), axis=1)
Dir10_Wid_Diff['Date'] = pd.to_datetime(Dir10_Wid_Diff['Date'], format ='%Y-%m-%d')

# Clean up and Fix Index
Dir10_Wid_Diff = Dir10_Wid_Diff[['Date','Month','Assgn_dam', 'Up_Avg_Wid', 'Ds_Avg_Wid', 'Width_Diff' ]]
Dir10_Wid_Diff

In [ ]:
## Export the data
Dir10_Wid_Diff.to_csv(r'F:\Insert_File_Output_Path_Here\Dir_Widths_byDam_DI.csv') # Update this file path

In [ ]:
##############################
##### Pull in the Daymet Data #####
##############################

In [ ]:
Daymet_Filepath = r"F:\Insert_File_Path_Here\Daymet"

# Get the list of all files
CSVFiles = glob.glob(os.path.join(Daymet_Filepath, "*.csv")) 

# Loop through the files for each dam and make one dataframe
Daymet_Data = pd.DataFrame()
for i in tqdm(range(len(CSVFiles))):
    try:
        x = pd.read_csv(CSVFiles[i], engine='python')
        Daymet_Data = pd.concat([Daymet_Data,x],axis=0)
    except pd.errors.EmptyDataError:
        print(CSVFiles[i], " is empty and has been skipped.") # In case some of the CSV files are empty

# Preview the data
Daymet_Data

In [ ]:
############ Get the Directly Upstream Daymet Average Info ############

In [ ]:
## Get a list of profiles to help with filtering
Profile_Details_Dir = Profile_Nodes_Daily.groupby(['Assgn_dam', 'Date']).agg({"Assgn_dam":'count'}) # doing a grouby to get the unique instances
Profile_Details_Dir.columns = ["Node_Count"] 

### Select the Daymet values for our dams of interest
DM_Dam_Select = Daymet_Data[Daymet_Data['Assgn_dam'].isin(Final_Dams_List)]
DM_Dam_Select['Date'] = pd.to_datetime(DM_Dam_Select['date'], format ='%Y%m%d')

# Get the days of interest for the profiles used
DM_Data_Filtered = pd.merge(DM_Dam_Select, Profile_Details_Dir, on= ['Date','Assgn_dam'], how = "inner")

# Get Up/Ds 
DM_Data_Filtered["Up_Ds"] = np.nan
DM_Data_Filtered["Up_Ds"] = np.where((DM_Data_Filtered['Dam_Dist'] > 0) , 'Downstream', DM_Data_Filtered["Up_Ds"])
DM_Data_Filtered["Up_Ds"] = np.where((DM_Data_Filtered['Dam_Dist'] < 0) , 'Upstream', DM_Data_Filtered["Up_Ds"])

# Get Up/Ds waterbody type
DM_Data_Filtered['Up_Ds_Grp']  = np.nan
DM_Data_Filtered['Up_Ds_Grp'] = np.where((DM_Data_Filtered['Up_Ds'] == 'Downstream') & (DM_Data_Filtered['lakeflag'] == 0) , 'Downstream River', DM_Data_Filtered['Up_Ds_Grp'])
DM_Data_Filtered['Up_Ds_Grp'] = np.where((DM_Data_Filtered['Up_Ds'] == 'Downstream') & (DM_Data_Filtered['lakeflag'] > 0) , 'Downstream Reservoir', DM_Data_Filtered['Up_Ds_Grp'])
DM_Data_Filtered['Up_Ds_Grp'] = np.where((DM_Data_Filtered['Up_Ds'] == 'Upstream') & (DM_Data_Filtered['lakeflag'] == 0) , 'Upstream River', DM_Data_Filtered['Up_Ds_Grp'])
DM_Data_Filtered['Up_Ds_Grp'] = np.where((DM_Data_Filtered['Up_Ds'] == 'Upstream') & (DM_Data_Filtered['lakeflag'] > 0) , 'Upstream Reservoir', DM_Data_Filtered['Up_Ds_Grp'])

# Get the Dam Dist in km
DM_Data_Filtered['Dam_Dist_km'] = DM_Data_Filtered['Dam_Dist'] /1000

In [ ]:
# Get the upstream average (10km)
DM_10km = DM_Data_Filtered[(DM_Data_Filtered['Dam_Dist_km'] >= -10) & (DM_Data_Filtered['Dam_Dist_km'] <= 0)]
DM_10km_avg = DM_10km.groupby(['Assgn_dam', 'Date']).agg({"srad": 'mean', "tmax": 'mean', "vp": 'mean' })
DM_10km_avg.columns = ["AvgUp_Rad", "AvgUp_TMax", "AvgUp_VP"]
DM_10km_avg = DM_10km_avg.reset_index()

# Preview
DM_10km_avg

In [ ]:
## Export Stuff
DM_10km_avg.to_csv(r"F:\Insert_File_Output_Path_Here\Daymet_Dir.csv") # Update this file path

In [ ]:
################################
##### Pull in the Discharge Data #####
################################

In [ ]:
GeoGlows_AvgQ = pd.read_csv(r"F:\Insert_File_Path_Here\Avg_Q_by_Dam.csv", engine = 'python') # Update this file path
GeoGlows_AvgQ = GeoGlows_AvgQ[['Assgn_dam', 'Date','Avg_Q']]
GeoGlows_AvgQ['Date'] =  pd.to_datetime(GeoGlows_AvgQ['Date'], format ='%Y-%m-%d')
GeoGlows_AvgQ

In [ ]:
###############################
### Create the Databeases for ML ###
###############################

In [ ]:
#### Directly Upstream -- 10 km ####

In [ ]:
## Dam Info 
Selected_Dams_Info = Selected_Dams[['DamID', 'Primary Purpose','NID Height (Ft)', 'Normal Storage (Acre-Ft)', 'Surface Area (Acres)', 'Drainage Area (Sq Miles)', 'Max Discharge (Cubic Ft/Second)', 'Resv']]
# Join Dam Info with Changes
Change_DamInfo_DIR = pd.merge(Avg_Diff_DIR, Selected_Dams_Info, left_on= ['Assgn_dam'], right_on = ['DamID'], how = "left")
Change_DamInfo_DIR = Change_DamInfo_DIR.drop(['Unnamed: 0','DamID'], axis=1)
Change_DamInfo_DIR['Date'] =  pd.to_datetime(Change_DamInfo_DIR['Date'], format ='%Y-%m-%d')

# Width Info
Width_Diff_Dir = Dir10_Wid_Diff[['Assgn_dam','Date', 'Width_Diff']]

# Meterological
RTMA_Attrib_Dir10 = RTMA_10km_avg[:]
Caravan_Daily_Dir10 =Daily_Dam_WS_Att_Prof[[ 'Assgn_dam', 'Date', 'volumetric_soil_water_layer_1_mean']]

# Basin Attributes
Caravan_Attrib_Dir10 = Dam_WS_Att_All[['DamID', 'pet_mean', 'aridity', 'high_prec_freq', 'high_prec_dur', 'dis_m3_pyr',
                                        'run_mm_syr', 'inu_pc_smx', 'dor_pc_pva', 'gwt_cm_sav', 'ele_mt_sav', 'for_pc_sse', 'urb_pc_sse', 
                                          'pre_mm_s01', 'pre_mm_s02', 'pre_mm_s03', 'pre_mm_s04', 'pre_mm_s05', 'pre_mm_s06', 'pre_mm_s07', 'pre_mm_s08', 'pre_mm_s09',  'pre_mm_s10', 'pre_mm_s11', 'pre_mm_s12', 
                                          'pet_mm_s01','pet_mm_s02', 'pet_mm_s03', 'pet_mm_s04', 'pet_mm_s05', 'pet_mm_s06','pet_mm_s07', 'pet_mm_s08', 'pet_mm_s09', 'pet_mm_s10', 'pet_mm_s11', 'pet_mm_s12', 
                                          'swc_pc_s01', 'swc_pc_s02', 'swc_pc_s03', 'swc_pc_s04', 'swc_pc_s05', 'swc_pc_s06', 'swc_pc_s07', 'swc_pc_s08', 'swc_pc_s09', 'swc_pc_s10', 'swc_pc_s11', 'swc_pc_s12']]

## Begin joining the data together 
Dir_Join1 = pd.merge(Change_DamInfo_DIR, Width_Diff_Dir, on=['Assgn_dam', 'Date'], how= 'left')
Dir_Join2 = pd.merge(Dir_Join1,RTMA_Attrib_Dir10, on=['Assgn_dam', 'Date'], how= 'left')
Dir_Join3 = pd.merge(Dir_Join2,Caravan_Daily_Dir10, on=['Assgn_dam', 'Date'], how = 'left')
Dir_Join4 = pd.merge(Dir_Join3, DM_10km_avg,  on=['Assgn_dam', 'Date'], how = 'left')
Dir_Join5 = pd.merge(Dir_Join4, GeoGlows_AvgQ, on=['Assgn_dam', 'Date'], how = 'left' )
Change_Attrib_Dir10 = pd.merge(Dir_Join5, Caravan_Attrib_Dir10, left_on= ["Assgn_dam"], right_on = ['DamID'], how = 'left') ## Note monthly columns will need filtering in ML steps
Change_Attrib_Dir10 = Change_Attrib_Dir10.drop(['DamID'], axis=1)

# Export the CSV File
Change_Attrib_Dir10.to_csv(r"F:\Insert_File_Output_Path_Here\All_Change_Attributes_Dir10.csv") # Update this file path

# Preview
Change_Attrib_Dir10

In [ ]:
## All Variable Options
print(Change_Attrib_Dir10.columns.tolist())

In [ ]:
### Get Seasonal Averages ###
Seas_Change_Attrib_Dir10 = Change_Attrib_Dir10[:]

# Precip
Seas_Change_Attrib_Dir10["Seasonal_PRE_Avg"] = np.nan
Seas_Change_Attrib_Dir10["Seasonal_PRE_Avg"] =np.where(Seas_Change_Attrib_Dir10["Month"].isin([12,1,2]), (Seas_Change_Attrib_Dir10['pre_mm_s12'] +  Seas_Change_Attrib_Dir10['pre_mm_s01'] +  Seas_Change_Attrib_Dir10['pre_mm_s02'])/3, Seas_Change_Attrib_Dir10["Seasonal_PRE_Avg"])
Seas_Change_Attrib_Dir10["Seasonal_PRE_Avg"] =np.where(Seas_Change_Attrib_Dir10["Month"].isin([3, 4, 5]), (Seas_Change_Attrib_Dir10['pre_mm_s03'] +  Seas_Change_Attrib_Dir10['pre_mm_s04'] +  Seas_Change_Attrib_Dir10['pre_mm_s05'])/3, Seas_Change_Attrib_Dir10["Seasonal_PRE_Avg"])
Seas_Change_Attrib_Dir10["Seasonal_PRE_Avg"] =np.where(Seas_Change_Attrib_Dir10["Month"].isin([6, 7, 8]), (Seas_Change_Attrib_Dir10['pre_mm_s06'] +  Seas_Change_Attrib_Dir10['pre_mm_s07'] +  Seas_Change_Attrib_Dir10['pre_mm_s08'])/3, Seas_Change_Attrib_Dir10["Seasonal_PRE_Avg"])
Seas_Change_Attrib_Dir10["Seasonal_PRE_Avg"] =np.where(Seas_Change_Attrib_Dir10["Month"].isin([9, 10, 11]), (Seas_Change_Attrib_Dir10['pre_mm_s09'] +  Seas_Change_Attrib_Dir10['pre_mm_s10'] +  Seas_Change_Attrib_Dir10['pre_mm_s11'])/3, Seas_Change_Attrib_Dir10["Seasonal_PRE_Avg"])

# PET
Seas_Change_Attrib_Dir10["Seasonal_PET_Avg"] = np.nan
Seas_Change_Attrib_Dir10["Seasonal_PET_Avg"] =np.where(Seas_Change_Attrib_Dir10["Month"].isin([12,1,2]), (Seas_Change_Attrib_Dir10['pet_mm_s12'] +  Seas_Change_Attrib_Dir10['pet_mm_s01'] +  Seas_Change_Attrib_Dir10['pet_mm_s02'])/3, Seas_Change_Attrib_Dir10["Seasonal_PET_Avg"])
Seas_Change_Attrib_Dir10["Seasonal_PET_Avg"] =np.where(Seas_Change_Attrib_Dir10["Month"].isin([3, 4, 5]), (Seas_Change_Attrib_Dir10['pet_mm_s03'] +  Seas_Change_Attrib_Dir10['pet_mm_s04'] +  Seas_Change_Attrib_Dir10['pet_mm_s05'])/3, Seas_Change_Attrib_Dir10["Seasonal_PET_Avg"])
Seas_Change_Attrib_Dir10["Seasonal_PET_Avg"] =np.where(Seas_Change_Attrib_Dir10["Month"].isin([6, 7, 8]), (Seas_Change_Attrib_Dir10['pet_mm_s06'] +  Seas_Change_Attrib_Dir10['pet_mm_s07'] +  Seas_Change_Attrib_Dir10['pet_mm_s08'])/3, Seas_Change_Attrib_Dir10["Seasonal_PET_Avg"])
Seas_Change_Attrib_Dir10["Seasonal_PET_Avg"] =np.where(Seas_Change_Attrib_Dir10["Month"].isin([9, 10, 11]), (Seas_Change_Attrib_Dir10['pet_mm_s09'] +  Seas_Change_Attrib_Dir10['pet_mm_s10'] +  Seas_Change_Attrib_Dir10['pet_mm_s11'])/3, Seas_Change_Attrib_Dir10["Seasonal_PET_Avg"])

# SWC
Seas_Change_Attrib_Dir10["Seasonal_SWC_Avg"] = np.nan
Seas_Change_Attrib_Dir10["Seasonal_SWC_Avg"] =np.where(Seas_Change_Attrib_Dir10["Month"].isin([12,1,2]), (Seas_Change_Attrib_Dir10['swc_pc_s12'] +  Seas_Change_Attrib_Dir10['swc_pc_s01'] +  Seas_Change_Attrib_Dir10['swc_pc_s02'])/3, Seas_Change_Attrib_Dir10["Seasonal_SWC_Avg"])
Seas_Change_Attrib_Dir10["Seasonal_SWC_Avg"] =np.where(Seas_Change_Attrib_Dir10["Month"].isin([3, 4, 5]), (Seas_Change_Attrib_Dir10['swc_pc_s03'] +  Seas_Change_Attrib_Dir10['swc_pc_s04'] +  Seas_Change_Attrib_Dir10['swc_pc_s05'])/3, Seas_Change_Attrib_Dir10["Seasonal_SWC_Avg"])
Seas_Change_Attrib_Dir10["Seasonal_SWC_Avg"] =np.where(Seas_Change_Attrib_Dir10["Month"].isin([6, 7, 8]), (Seas_Change_Attrib_Dir10['swc_pc_s06'] +  Seas_Change_Attrib_Dir10['swc_pc_s07'] +  Seas_Change_Attrib_Dir10['swc_pc_s08'])/3, Seas_Change_Attrib_Dir10["Seasonal_SWC_Avg"])
Seas_Change_Attrib_Dir10["Seasonal_SWC_Avg"] =np.where(Seas_Change_Attrib_Dir10["Month"].isin([9, 10, 11]), (Seas_Change_Attrib_Dir10['swc_pc_s09'] +  Seas_Change_Attrib_Dir10['swc_pc_s10'] +  Seas_Change_Attrib_Dir10['swc_pc_s11'])/3, Seas_Change_Attrib_Dir10["Seasonal_SWC_Avg"])

Seas_Change_Attrib_Dir10

In [ ]:
### Clean Version -- Dir ###
Change_Attrib_Dir10_Clean = Seas_Change_Attrib_Dir10.drop(['Up_Avg', 'Ds_Avg', 'Abs_Temp_Diff',
'pre_mm_s01', 'pre_mm_s02', 'pre_mm_s03', 'pre_mm_s04', 'pre_mm_s05', 'pre_mm_s06', 'pre_mm_s07', 'pre_mm_s08', 'pre_mm_s09', 'pre_mm_s10', 'pre_mm_s11', 'pre_mm_s12',
'pet_mm_s01', 'pet_mm_s02', 'pet_mm_s03', 'pet_mm_s04', 'pet_mm_s05', 'pet_mm_s06', 'pet_mm_s07', 'pet_mm_s08', 'pet_mm_s09', 'pet_mm_s10', 'pet_mm_s11', 'pet_mm_s12', 
'swc_pc_s01', 'swc_pc_s02', 'swc_pc_s03', 'swc_pc_s04', 'swc_pc_s05', 'swc_pc_s06', 'swc_pc_s07', 'swc_pc_s08', 'swc_pc_s09', 'swc_pc_s10', 'swc_pc_s11', 'swc_pc_s12'], axis=1)

## Remove the Values After Daymet Ends in 2024
Change_Attrib_Dir10_Clean = Change_Attrib_Dir10_Clean[Change_Attrib_Dir10_Clean['Date'] <= '2024-12-31']

## Fix Nulls ##
# Get a copy of the data
FixNull1 = Change_Attrib_Dir10_Clean[:]

# Fix Categorical Nulls
FixNull1 = FixNull1.fillna({'Primary Purpose': "Unknown"})

# Add Seasons -- We want to fix with seasonal medians
FixNull1["Season"] = np.nan
FixNull1["Season"] =np.where(FixNull1["Month"].isin([12,1,2]), 'Winter', FixNull1["Season"])
FixNull1["Season"] =np.where(FixNull1["Month"].isin([3, 4, 5]), 'Spring', FixNull1["Season"])
FixNull1["Season"] =np.where(FixNull1["Month"].isin([6, 7, 8]), 'Summer', FixNull1["Season"])
FixNull1["Season"] =np.where(FixNull1["Month"].isin([9, 10, 11]),'Fall', FixNull1["Season"])

# lists for loops
damlist = FixNull1['Assgn_dam'].unique().tolist()
seasonlist = ['Winter','Spring', 'Summer','Fall']

# Create new dataframe for storing
Change_Attrib_Dir10_Clean_NA = pd.DataFrame()

## For Replacing by the Dam Median
for i in damlist:
    FixNull2 = FixNull1[FixNull1['Assgn_dam'] == i]
    for col in FixNull2.columns:
        if FixNull2[col].isnull().any(): # Check if there are any NaNs in the column
            FixNull2[col].fillna(FixNull2[col].median(), inplace=True)
            Output = pd.concat([Change_Attrib_Dir10_Clean_NA, FixNull2])
        else:
            Output = pd.concat([Change_Attrib_Dir10_Clean_NA, FixNull2])  
    Change_Attrib_Dir10_Clean_NA = Output[:]

In [ ]:
###### Get the date information and do a sine/cosine transform ######
# Get a clean copy of the df
Change_Attrib_Dir10_SinCos = Change_Attrib_Dir10_Clean_NA[:]

# Get day of year and leap year information
Change_Attrib_Dir10_SinCos["DOY"] = Change_Attrib_Dir10_SinCos['Date'].dt.dayofyear
Change_Attrib_Dir10_SinCos["LeapYear"] = Change_Attrib_Dir10_SinCos['Date'].dt.is_leap_year
Change_Attrib_Dir10_SinCos["NumDays"] = np.where(Change_Attrib_Dir10_SinCos["LeapYear"] == True, 366, 365)

# Transform the Data
Change_Attrib_Dir10_SinCos["DOY_Sin"] = np.sin(Change_Attrib_Dir10_SinCos["DOY"] *(2.*np.pi/Change_Attrib_Dir10_SinCos["NumDays"]))
Change_Attrib_Dir10_SinCos["DOY_Cos"] = np.cos(Change_Attrib_Dir10_SinCos["DOY"] *(2.*np.pi/Change_Attrib_Dir10_SinCos["NumDays"]))

# Preview the Data
Change_Attrib_Dir10_SinCos

In [ ]:
################# Get the Physiographic Regions #################

In [ ]:
### Pull in Physio Regions ###
Physio = gpd.read_file(r"F:\Insert_File_Path_Here\Physiographic_Regions.shp", columns =['PROVINCE']) # Update this file path

### Get the geodatabase version of Grod ###
GrodDams = gpd.read_file(r"F:\Insert_File_Path_Here\Study_Dams.shp", columns=['grod_id', 'type', 'lon', 'lat', 'hilarriid', 'dataset']) # Update this file path

## Get the list of Damsdam in current dataset ##
Current_ML_Dams = Change_Attrib_Dir10_SinCos['Assgn_dam'].unique().tolist()
GrodDams_Filtered = GrodDams[GrodDams['grod_id'].isin(Current_ML_Dams)]

# Join the sampled dams to their intersecting Physiographic Regions (8)
DamPhysioJoin = gpd.sjoin(Physio, GrodDams_Filtered, how='inner', predicate='contains') 
DamPhysioJoin = DamPhysioJoin.reset_index()
DamPhysioJoin = DamPhysioJoin[['grod_id', 'PROVINCE']]

# Fix the two that have boundary issues with the polygons
new_rows_data = {'grod_id': [311.0, 416.0], 'PROVINCE': ['SUPERIOR UPLAND', 'CENTRAL LOWLAND']}
new_rows_df = pd.DataFrame(new_rows_data)
DamPhysioJoin_updated = pd.concat([DamPhysioJoin, new_rows_df], ignore_index=True)

### Join Region with other Dam observations ###
Change_Physio = pd.merge(Change_Attrib_Dir10_SinCos, DamPhysioJoin_updated, left_on = 'Assgn_dam', right_on ='grod_id', how = 'left')
Change_Physio

In [ ]:
##### Clean up the Column Names For R and Graphics #####
# Drop unused columns
Clean_ML_Inputs = Change_Physio.drop(['YearMo','DOY','LeapYear','NumDays','Season', 'grod_id'], axis=1)

# Rename Columns
Clean_ML_Inputs = Clean_ML_Inputs.rename(columns={'Primary Purpose': 'Dam purpose', 'NID Height (Ft)': 'Dam height', 'Normal Storage (Acre-Ft)': 'Dam storage', 'Surface Area (Acres)': 'Surface area',
                                                 'Drainage Area (Sq Miles)': 'Drainage area', 'Max Discharge (Cubic Ft/Second)': 'Max discharge', 'Width_Diff': 'Up/Downstream width difference', 'AvgUp_Wind': 'Wind',
                                                  'volumetric_soil_water_layer_1_mean': 'Soil water content', 'AvgUp_Rad': 'Shortwave radiation', 'AvgUp_TMax' : 'Daily maximum air temperature',
                                                  'AvgUp_VP': 'Vapor pressure', 'Avg_Q': 'Discharge', 'pet_mean': 'Annual mean PET', 'aridity': 'Aridity', 'high_prec_freq': 'High precipitation frequency',
                                                   'high_prec_dur': 'High precipitation duration', 'dis_m3_pyr': 'Annual watershed discharge', 'run_mm_syr': 'Annual watershed runoff', 
                                                   'inu_pc_smx': 'Inundation extent', 'dor_pc_pva': 'Degree of regulation', 'gwt_cm_sav': 'Groundwater table depth', 'ele_mt_sav': 'Elevation', 
                                                   'for_pc_sse': 'Forest extent', 'urb_pc_sse': 'Urban extent', 'Seasonal_PRE_Avg': 'Seasonal Precipitation', 'Seasonal_PET_Avg': 'Seasonal PET', 
                                                   'Seasonal_SWC_Avg': 'Seasonal soil water content', 'PROVINCE': 'Physiographic Region'})

In [ ]:
### Export the CSV File ###
# Export
Clean_ML_Inputs.to_csv(r"F:\Insert_File_Output_Path_Here\Temp_Change_Attributes_All.csv") # Update this file path

# Preview
Clean_ML_Inputs

In [ ]:
######### Final Stats #########

In [ ]:
Dam_Dets = Clean_ML_Inputs[Clean_ML_Inputs['Sig_05'] == "Significant"]
Dam_Dets = Dam_Dets.dropna()
print("Total Dams: ", len(Dam_Dets['Assgn_dam'].unique().tolist()))
print("Total Profiles: ", len(Dam_Dets))

In [ ]:
## Save the Final Dam Lists
Dam_Dets_List = Dam_Dets['Assgn_dam'].unique().tolist()
Dam_ML = pd.DataFrame(Dam_Dets_List, columns=['DamID'])
Dam_ML.to_csv(r"F:\Insert_File_Output_Path_Here\Final_Dam_List.csv") # Update this file path

In [ ]:
print(Dam_ML['DamID'].tolist())